<a href="https://colab.research.google.com/github/yashdeepspodder23/BlackHoleSimulations/blob/main/BHSim1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import collections

# Constants
G = 1  # Gravitational constant (geometrized units)
M = 100  # Mass of black hole
dt = 0.01  # Time step
num_p = 50  # Number of particles

# Initialize particle positions and velocities
np.random.seed(100)
poss = np.random.uniform(-10, 10, (num_p, 2))
vel = np.zeros((num_p, 2))

# Black hole position
bh_pos = np.array([0, 0])

# Global variables for trail
trail_length = 50 # How many previous positions to store for the trail
# Initialize an empty list of deques, one deque per particle to store its trail
trail_data = [collections.deque(maxlen=trail_length) for _ in range(num_p)]

# Gravitational force calculation
def grav_force(pos):
    r = np.linalg.norm(pos - bh_pos)
    if r < 1e-6:  # Avoid division by zero near the black hole
        return np.array([0, 0])
    f_mag = G * M / r**2
    dirn = (bh_pos - pos) / r
    return f_mag * dirn

# Update particle positions and velocities
def update_particles(poss, vel):
    for i in range(len(poss)):
        f = grav_force(poss[i])
        vel[i] += f * dt
        poss[i] += vel[i] * dt
    return poss, vel

# Set up plot
fig, ax = plt.subplots()
ax.set_xlim(-15, 15)
ax.set_ylim(-15, 15)
ax.set_facecolor('black') # Set background to black for space-like visual

# Add some static 'stars' in the background
num_stars = 200
star_x = np.random.uniform(-15, 15, num_stars)
star_y = np.random.uniform(-15, 15, num_stars)
ax.scatter(star_x, star_y, s=1, color='white', alpha=0.7, zorder=0) # zorder to keep stars behind particles

# Add an accretion disk
# Choose a radius for the disk. Adjust as needed.
disk_radius = 2.5 # Example radius
accretion_disk = plt.Circle(bh_pos, disk_radius, color='orange', alpha=0.6, zorder=1, label='Accretion Disk')
ax.add_patch(accretion_disk)

bh = plt.scatter(*bh_pos, color='black', s=100, label='Black Hole', zorder=2)

# Create a list of Line2D objects for each particle's trail
particles_trails = []
# Generate distinct colors for each particle
colors = plt.cm.jet(np.linspace(0, 1, num_p))
for i in range(num_p):
    # Changed 'o-' to 'o:' for dotted lines, markersize increased slightly for visibility
    if i == 0:
        line, = ax.plot([], [], 'o:', color=colors[i], markersize=3, alpha=0.7, label='Particles', zorder=1.5) # Zorder updated
    else:
        line, = ax.plot([], [], 'o:', color=colors[i], markersize=3, alpha=0.7, zorder=1.5) # Zorder updated
    particles_trails.append(line)

ax.legend()

# Init function
def init():
    for line in particles_trails:
        line.set_data([], [])
    return particles_trails # Return all changing artists

# Animation function
def animate(frame):
    global poss, vel
    poss, vel = update_particles(poss, vel)

    # Update trail data for each particle
    for i in range(num_p):
        trail_data[i].append(poss[i]) # Append current position

    artists_to_return = []
    # Update each particle's trail line
    for i, line in enumerate(particles_trails):
        # Extract x and y coordinates from the deque
        # Each position in deque is a numpy array [x, y]
        x_coords = [p[0] for p in trail_data[i]]
        y_coords = [p[1] for p in trail_data[i]]
        line.set_data(x_coords, y_coords)
        artists_to_return.append(line)

    return artists_to_return # Return all updated artists (the particle trails)

# Create animation
ani = FuncAnimation(fig, animate, frames=1000, init_func=init, interval=20, blit=True)
plt.title('Black Hole Simulation with Accretion Disk using Newtonian gravity') # Updated title
#plt.show()

from IPython.display import HTML

# Convert the animation to an HTML5 video
html_animation = HTML(ani.to_html5_video())

# Display the animation
display(html_animation)